# 🕸️ LangGraph Fundamentals: StateGraph, Nodes & Conditional Routing

### Overview & Learning Objectives
LangGraph is a framework for building stateful, multi-actor applications with LLMs, modeling workflows as **directed graphs**.

At its core are three fundamental concepts:
1. **State**: The shared data structure that passes between nodes, defined as a typed dictionary or Pydantic model.
2. **Nodes**: Python functions representing actions, tools, or model calls that receive the current state and return state updates.
3. **Edges**: Control-flow rules connecting nodes, either **fixed** (always proceed to target node) or **conditional** (dynamically route based on state values).

## 📐 System Architecture: The Simplest Graph

The diagram below illustrates the StateGraph topology with a probabilistic branch:

<div align="center">
  <img src="images/05_stategraph_core_nodes.png" alt="LangGraph StateGraph Core Nodes & Conditional Branching" width="100%" />
</div>

<br/>

<details>
<summary><b>🔍 View Raw Mermaid Diagram Syntax</b></summary>

```mermaid
flowchart TD
    START([🚀 START]) --> N1["🟦 Node 1<br/>Appends ' I am'"]
    N1 --> Branch{"🔀 decide_mood()<br/>Evaluates State"}
    Branch -->|50% Probability| N2["🟩 Node 2<br/>Appends ' happy!'"]
    Branch -->|50% Probability| N3["🟥 Node 3<br/>Appends ' sad!'"]
    N2 --> END([🏁 END])
    N3 --> END
```

</details>


## 1. Installation

Install LangGraph and core dependencies.

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain_core

## 2. Defining the State Schema

The **State** schema serves as the interface contract for all nodes and edges. We use Python's `TypedDict` to declare the structure and type hints for all keys stored in the graph.

- Keys represent **state channels**.
- By default, returning a dictionary with an existing key will **overwrite** its previous value.

In [ ]:
from typing_extensions import TypedDict

class State(TypedDict):
    graph_state: str

## 3. Defining Nodes

Nodes are standard Python functions:
- **Input**: Receives the current `State` dictionary as its primary argument.
- **Output**: Returns a dictionary specifying the key updates to apply back to the state.

Below, `node_1`, `node_2`, and `node_3` mutate the `graph_state` string.

In [ ]:
def node_1(state: State):
    print("--- Executing Node 1 ---")
    return {"graph_state": state['graph_state'] + " I am"}

def node_2(state: State):
    print("--- Executing Node 2 ---")
    return {"graph_state": state['graph_state'] + " happy!"}

def node_3(state: State):
    print("--- Executing Node 3 ---")
    return {"graph_state": state['graph_state'] + " sad!"}

## 4. Defining Edges & Conditional Routing

- **Normal Edges (`add_edge`)**: Form deterministic connections between nodes.
- **Conditional Edges (`add_conditional_edges`)**: Route between nodes based on dynamic logic. A conditional routing function inspects the state and returns the name of the next node as a `Literal` string.

In [ ]:
import random
from typing import Literal

def decide_mood(state: State) -> Literal["node_2", "node_3"]:
    """Conditional routing function that branches 50/50 between node_2 and node_3."""
    user_input = state['graph_state']
    
    # Probabilistic branch (in agents, this would inspect LLM output or tool calls)
    if random.random() < 0.5:
        return "node_2"
    return "node_3"

## 5. Building & Compiling the `StateGraph`

We assemble the graph components:
1. Instantiate `StateGraph(State)`.
2. Register nodes using `.add_node(name, fn)`.
3. Establish the starting edge from `START` to `node_1`.
4. Attach the conditional routing edge from `node_1`.
5. Connect terminal nodes (`node_2`, `node_3`) to `END`.
6. Call `.compile()` to validate topology and generate an executable runnable.

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

# 1. Initialize StateGraph with State schema
builder = StateGraph(State)

# 2. Add nodes
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 3. Add deterministic edge
builder.add_edge(START, "node_1")

# 4. Add conditional routing edge
builder.add_conditional_edges("node_1", decide_mood)

# 5. Add terminal edges
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# 6. Compile the graph
graph = builder.compile()
print("✅ StateGraph compiled successfully.")

# Visualize compiled graph via Mermaid PNG (falls back gracefully if rendering engine unavailable)
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Mermaid PNG rendering skipped: {e}")

## 6. Execution Lifecycle: Sequence Diagram

<div align="center">
  <img src="images/seq_stategraph_lifecycle.png" alt="StateGraph Execution Lifecycle Sequence Diagram" width="100%" />
</div>

<br/>

<details>
<summary><b>🔍 View Raw Mermaid Diagram Syntax</b></summary>

```mermaid
sequenceDiagram
    autonumber
    actor User
    participant Engine as LangGraph Pregel Engine
    participant StateStore as State Channel
    participant N1 as node_1
    participant Route as decide_mood
    participant N23 as node_2 / node_3

    User->>Engine: graph.invoke({'graph_state': 'Hi'}) 
    Engine->>StateStore: Initialize state
    Engine->>N1: Step 1: Execute node_1(state)
    N1-->>StateStore: Write {'graph_state': 'Hi I am'}
    Engine->>Route: Step 2: Evaluate decide_mood(state)
    Route-->>Engine: Returns target node name
    Engine->>N23: Step 3: Execute node_2 or node_3(state)
    N23-->>StateStore: Write updated state string
    Engine->>Engine: Reach END condition
    Engine-->>User: Return final state dictionary
```
</details>

## 7. Invoking the Graph Synchronously

`graph.invoke(...)` executes the graph from `START` to `END` synchronously, returning the final state dictionary.

In [ ]:
result = graph.invoke({"graph_state": "Hi, this is Lance."})
print("\nFinal Graph Output:")
print(result)

## 8. Streaming State Transitions

LangGraph supports streaming intermediate superstep events with `.stream()`, enabling step-by-step progress tracking for UI applications.

In [ ]:
print("Streaming execution events:")
for event in graph.stream({"graph_state": "Streaming test:"}):
    print("Event:", event)